In [ ]:
"""
第3部分：向量化和语义检索（RAG核心）
"""

from sentence_transformers import SentenceTransformer
import numpy as np
from elasticsearch import Elasticsearch
import json


class VectorSearcher:
    """向量语义检索器"""
    
    def __init__(self, es, index_name='medical_guidelines', 
                 model_name='paraphrase-multilingual-MiniLM-L12-v2'):
        """
        初始化向量检索器
        
        model_name可选:
        - paraphrase-multilingual-MiniLM-L12-v2 (多语言，轻量)
        - shibing624/text2vec-base-chinese (中文专用)
        """
        self.es = es
        self.index_name = index_name
        
        # 加载embedding模型
        print(f"正在加载模型: {model_name}...")
        self.model = SentenceTransformer(model_name)
        print("✅ 模型加载完成")
    
    # ==================== 1. 为文档生成向量 ====================
    def add_vectors_to_documents(self, json_file):
        """为所有文档生成向量并更新到ES"""
        
        # 读取文档
        with open(json_file, 'r', encoding='utf-8') as f:
            documents = json.load(f)
        
        print(f"正在为 {len(documents)} 条文档生成向量...")
        
        # 为每个文档生成向量
        for i, doc in enumerate(documents):
            # 构建用于向量化的文本
            text_for_embedding = self._build_text_for_embedding(doc)
            
            # 生成向量
            vector = self.model.encode(text_for_embedding)
            
            # 更新文档（添加vector字段）
            self.es.update(
                index=self.index_name,
                id=doc['rule_id'],
                body={
                    "doc": {
                        "vector": vector.tolist()
                    }
                }
            )
            
            if (i + 1) % 10 == 0:
                print(f"  已处理 {i + 1}/{len(documents)} 条")
        
        print("✅ 向量生成完成")
    
    def _build_text_for_embedding(self, doc):
        """构建用于向量化的文本（综合关键信息）"""
        
        parts = []
        
        # 药物名称
        drug_name = doc['intervention']['name']
        parts.append(f"药物: {drug_name}")
        
        # 分类
        if doc['intervention'].get('category'):
            parts.append(f"分类: {doc['intervention']['category']}")
        
        # 治疗阶段
        parts.append(f"阶段: {doc['phase']}")
        
        # 给药途径
        if doc['intervention']['route'].get('description'):
            parts.append(f"给药: {doc['intervention']['route']['description']}")
        
        # 适应症（从原始文本中提取）
        if doc['traceability'].get('raw_text_segment'):
            raw_text = doc['traceability']['raw_text_segment'][:200]
            parts.append(raw_text)
        
        return " ".join(parts)
    
    # ==================== 2. 向量相似度搜索 ====================
    def vector_search(self, query_text, size=10, min_score=0.5):
        """基于向量的语义搜索"""
        
        # 将查询文本向量化
        query_vector = self.model.encode(query_text)
        
        # ES脚本查询（计算余弦相似度）
        script_query = {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.query_vector, 'vector') + 1.0",
                    "params": {"query_vector": query_vector.tolist()}
                }
            }
        }
        
        query = {
            "size": size,
            "query": script_query,
            "_source": {"excludes": ["vector"]}  # 不返回vector字段
        }
        
        result = self.es.search(index=self.index_name, body=query)
        
        # 过滤低相似度结果
        filtered_results = []
        for hit in result['hits']['hits']:
            # ES的script_score返回的分数 = 相似度 + 1.0
            similarity = hit['_score'] - 1.0
            if similarity >= min_score:
                hit['similarity'] = similarity
                filtered_results.append(hit)
        
        return self._format_vector_results(filtered_results, result['hits']['total']['value'])
    
    # ==================== 3. 混合检索（关键词+向量）====================
    def hybrid_search(self, query_text, size=10, keyword_weight=0.3, vector_weight=0.7):
        """混合检索：结合关键词匹配和语义相似度"""
        
        # 1. 关键词搜索
        keyword_query = {
            "multi_match": {
                "query": query_text,
                "fields": [
                    "intervention.name^3",
                    "intervention.category^2",
                    "traceability.raw_text_segment"
                ]
            }
        }
        
        # 2. 向量搜索
        query_vector = self.model.encode(query_text)
        vector_script = {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.query_vector, 'vector') + 1.0",
                    "params": {"query_vector": query_vector.tolist()}
                }
            }
        }
        
        # 执行两次查询并合并结果
        keyword_results = self.es.search(
            index=self.index_name,
            body={"query": keyword_query, "size": size * 2}
        )
        
        vector_results = self.es.search(
            index=self.index_name,
            body={"query": vector_script, "size": size * 2}
        )
        
        # 合并并重新排序
        merged = self._merge_results(
            keyword_results['hits']['hits'],
            vector_results['hits']['hits'],
            keyword_weight,
            vector_weight
        )
        
        return merged[:size]
    
    def _merge_results(self, keyword_hits, vector_hits, kw_weight, vec_weight):
        """合并关键词和向量检索结果"""
        
        # 使用字典存储合并后的分数
        merged_scores = {}
        
        # 关键词结果
        for hit in keyword_hits:
            doc_id = hit['_id']
            merged_scores[doc_id] = {
                'keyword_score': hit['_score'],
                'vector_score': 0,
                'doc': hit
            }
        
        # 向量结果
        for hit in vector_hits:
            doc_id = hit['_id']
            if doc_id in merged_scores:
                merged_scores[doc_id]['vector_score'] = hit['_score'] - 1.0
            else:
                merged_scores[doc_id] = {
                    'keyword_score': 0,
                    'vector_score': hit['_score'] - 1.0,
                    'doc': hit
                }
        
        # 计算混合分数
        for doc_id in merged_scores:
            item = merged_scores[doc_id]
            # 归一化
            kw_normalized = item['keyword_score'] / 10 if item['keyword_score'] > 0 else 0
            vec_normalized = item['vector_score']
            
            item['hybrid_score'] = (kw_normalized * kw_weight + 
                                   vec_normalized * vec_weight)
        
        # 按混合分数排序
        sorted_results = sorted(
            merged_scores.values(),
            key=lambda x: x['hybrid_score'],
            reverse=True
        )
        
        # 格式化返回
        formatted = []
        for item in sorted_results:
            hit = item['doc']
            hit['_score'] = item['hybrid_score']
            hit['keyword_score'] = item['keyword_score']
            hit['vector_score'] = item['vector_score']
            formatted.append(hit)
        
        return formatted
    
    # ==================== 4. 格式化结果 ====================
    def _format_vector_results(self, hits, total):
        """格式化向量检索结果"""
        
        formatted = {
            'total': total,
            'results': []
        }
        
        for hit in hits:
            source = hit['_source']
            formatted_item = {
                'similarity': hit.get('similarity', hit['_score'] - 1.0),
                'drug_name': source['intervention']['name'],
                'category': source['intervention'].get('category', ''),
                'phase': source['phase'],
                'route': source['intervention']['route'].get('description', ''),
                'frequency': source['intervention'].get('frequency', ''),
                'evidence_level': source['evidence'].get('evidence_level', ''),
                'recommendation_grade': source['evidence'].get('recommendation_grade', ''),
                'source_guideline': source['source'].get('guideline', ''),
                'full_doc': source
            }
            formatted['results'].append(formatted_item)
        
        return formatted
    
    # ==================== 5. 打印结果 ====================
    def print_vector_results(self, results):
        """打印向量检索结果"""
        
        print(f"\n找到 {results['total']} 条结果")
        print("="*80)
        
        for i, item in enumerate(results['results'], 1):
            print(f"\n【结果 {i}】 相似度: {item['similarity']:.4f}")
            print(f"药物名称: {item['drug_name']}")
            print(f"药物分类: {item['category']}")
            print(f"治疗阶段: {item['phase']}")
            print(f"给药方式: {item['route']}")
            print(f"证据等级: {item['evidence_level']}")
            print(f"来源: {item['source_guideline']}")
            print("-"*80)


# ==================== 使用示例 ====================
if __name__ == "__main__":
    # 连接ES
    es = Elasticsearch([{'host': 'localhost', 'port': 9200, 'scheme': 'http'}])
    
    # 创建向量检索器
    vector_searcher = VectorSearcher(es)
    
    # 步骤1: 首次使用需要为文档生成向量（只需执行一次）
    # vector_searcher.add_vectors_to_documents('all_elasticsearch_docs.json')
    
    # 步骤2: 语义搜索示例
    print("\n【示例1】语义搜索: '偏头痛发作时应该吃什么药'")
    results = vector_searcher.vector_search("偏头痛发作时应该吃什么药", size=5)
    vector_searcher.print_vector_results(results)
    
    print("\n【示例2】语义搜索: '如何预防偏头痛'")
    results = vector_searcher.vector_search("如何预防偏头痛", size=5)
    vector_searcher.print_vector_results(results)
    
    print("\n【示例3】混合检索: '布洛芬治疗偏头痛'")
    results = vector_searcher.hybrid_search("布洛芬治疗偏头痛", size=5)
    vector_searcher.print_vector_results(results)